In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

# Ensure we can import from src/ or the current working directory
ROOT = Path().resolve()
if (ROOT / "data_processing.py").exists():
    sys.path.append(str(ROOT))
elif (ROOT / "src" / "data_processing.py").exists():
    sys.path.append(str(ROOT / "src"))

from data_processing import build_model_df


In [2]:
# ----------------------------
# Build model_df using shared pipeline
# ----------------------------
DATA_DIR = Path("data")
if not DATA_DIR.exists():
    alt_data = ROOT / "src" / "data"
    if alt_data.exists():
        DATA_DIR = alt_data

SAVE_MODEL_DF = True
CSV_PATH = DATA_DIR / "model_df.csv"

model_df = build_model_df(
    data_dir=DATA_DIR,
    save_csv=SAVE_MODEL_DF,
    csv_path=CSV_PATH,
)


Data processing is now centralized in `src/data_processing.py`.
This notebook uses `build_model_df(...)` to create `model_df`.


In [3]:
import numpy as np
import patsy
import statsmodels.formula.api as smf

# ----------------------------
# 8) Linear regression (robust version)
#   - attraction FE: C(ENTITY_DESCRIPTION_SHORT)
#   - DOW FE: C(dow)
#   - season FE: C(season)
#   - COVID interactions with utilization
#   - Clustered SEs by attraction, aligned to used rows
#   - Time-based train/test split (80/20, no shuffle)
# ----------------------------

# (Optional but recommended) fill some common missing merges so you don't lose lots of rows
if "attendance" in model_df.columns:
    model_df["attendance"] = model_df["attendance"].fillna(model_df["attendance"].median())
if "scheduled_open_min" in model_df.columns:
    model_df["scheduled_open_min"] = model_df["scheduled_open_min"].fillna(model_df["scheduled_open_min"].median())

# Weather terms (only include if column exists AND has at least some non-missing values)
weather_candidates = ["temp", "rain_1h", "wind_speed", "clouds_all", "humidity"]
weather_terms = [
    c for c in weather_candidates
    if c in model_df.columns and model_df[c].notna().any()
]

# Build formula pieces
base_terms = [
    "utilization",
    "availability",
    "np.log1p(attendance)",
    "scheduled_open_min",
    "nb_units_med",
    "covid",
    "post_covid",
    "utilization:covid",
    "utilization:post_covid",
    "C(dow)",
    "C(season)",
    "C(ENTITY_DESCRIPTION_SHORT)"
]

all_terms = base_terms + weather_terms
formula = "wait_time_avg ~ " + " + ".join(all_terms)

# ----------------------------
# Time-based split (no shuffle)
# ----------------------------
model_df_sorted = model_df.sort_values("date").reset_index(drop=True)
split_idx = int(len(model_df_sorted) * 0.8)

train_df = model_df_sorted.iloc[:split_idx].copy()
test_df = model_df_sorted.iloc[split_idx:].copy()

# Build design matrices on train to define columns
# (patsy will drop rows with missing values used in the formula)
y_train, X_train = patsy.dmatrices(formula, data=train_df, return_type="dataframe")
train_used_idx = X_train.index

print(f"Rows in model_df: {len(model_df_sorted):,}")
print(f"Rows used in train (after NA handling): {len(train_used_idx):,}")
print(f"Rows in test before NA handling: {len(test_df):,}")
print(f"Weather terms included: {weather_terms}")

# Align clustering groups to the used rows
# (clusters are based on attraction)
groups_train = train_df.loc[train_used_idx, "ENTITY_DESCRIPTION_SHORT"]

# Fit OLS with clustered SEs on train
fit = smf.ols(formula, data=train_df.loc[train_used_idx]).fit(
    cov_type="cluster",
    cov_kwds={"groups": groups_train}
)

print(fit.summary())

# ----------------------------
# Out-of-sample evaluation on test set
# ----------------------------
# Build test matrices and align columns to train
# (if any category levels are missing in test, fill with 0)
y_test, X_test = patsy.dmatrices(formula, data=test_df, return_type="dataframe")
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

test_pred = np.dot(X_test, fit.params)
test_actual = y_test.iloc[:, 0].to_numpy()

rmse = np.sqrt(np.mean((test_actual - test_pred) ** 2))
mae = np.mean(np.abs(test_actual - test_pred))

print(f"Test RMSE: {rmse:.4f}")
print(f"Test MAE: {mae:.4f}")


c:\Users\artur\Desktop\ESSEC MS IN DATA SCIENCE\0 STUDY\M2\Hackathon\code\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\artur\Desktop\ESSEC MS IN DATA SCIENCE\0 STUDY\M2\Hackathon\code\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


Rows in model_df: 29,847
Rows used in train (after NA handling): 23,283
Rows in test before NA handling: 5,970
Weather terms included: ['temp', 'rain_1h', 'wind_speed', 'clouds_all', 'humidity']
                            OLS Regression Results                            
Dep. Variable:          wait_time_avg   R-squared:                       0.742
Model:                            OLS   Adj. R-squared:                  0.742
Method:                 Least Squares   F-statistic:                 5.382e+08
Date:                Tue, 10 Feb 2026   Prob (F-statistic):           2.65e-99
Time:                        13:33:46   Log-Likelihood:                -85136.
No. Observations:               23283   AIC:                         1.704e+05
Df Residuals:                   23235   BIC:                         1.708e+05
Df Model:                          47                                         
Covariance Type:              cluster                                         
               

c:\Users\artur\Desktop\ESSEC MS IN DATA SCIENCE\0 STUDY\M2\Hackathon\code\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 47, but rank is 23
  warnings.warn('covariance of constraints does not have full '
